<a href="https://colab.research.google.com/github/muneeb0065/Machine-Learning/blob/main/Bagging_Ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

In [2]:
# Generate a Random Dataset
X,y = make_classification(n_samples=10000, n_features=10,n_informative=3)

In [3]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [4]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)
y_pred = dt.predict(X_test)

print("Decision Tree accuracy",accuracy_score(y_test,y_pred))

Decision Tree accuracy 0.904


As we can see the accuracy score comes 0.9.
Now we will apply bagging.

# **Bagging**

In [6]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.5,
    bootstrap=True,
    random_state=42
)

Here is the details of the hyperparameters of the above:

estimator means the base model we are using for our dataset.

n_estimators mean the number of model we are training. here we train 500 Desision Trees.

max_samples mean the max number of randomly selected rows for each model.

bootstrap True means we are performing sampling with replacement.

In [7]:
bag.fit(X_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.5,
                  n_estimators=500, random_state=42)

In [8]:
y_pred = bag.predict(X_test)

In [9]:
accuracy_score(y_test,y_pred)

0.9465

# **Bagging using SVM**

In [11]:
bag = BaggingClassifier(
    estimator=SVC(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42
)

In [12]:
bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Bagging using SVM",accuracy_score(y_test,y_pred))

Bagging using SVM 0.93


In [19]:
bag.estimators_samples_[0].shape

(8000,)

# **Pasting**

In [13]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False,
    random_state=42,
    verbose = 1,
    n_jobs=-1
)


In above cell we just set bootstrap = False means we are sampling without replacement.

Here n_jobs=-1 means we are utilizing all cores of the cpu for fast model training.

Verbose means we can see whats happening which you can see in output of next cell.

In [14]:
bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Pasting classifier",accuracy_score(y_test,y_pred))

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:   25.2s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


Pasting classifier 0.946


[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    0.4s finished


# **Random Subspaces**

In [16]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    bootstrap=False,
    max_features=0.5,
    bootstrap_features=False,
    random_state=42
)

Here we just use ***bootstrap_features=False***   means we are doing coloumns sampling without replacement also we set ***bootstrap=False***  means row sampling without sampling. because sciket learn uses defaul value which is ***True***

In [18]:
bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Random Subspaces classifier",accuracy_score(y_test,y_pred))

Random Subspaces classifier 0.937


In [21]:
bag.estimators_samples_[0].shape

(8000,)

In [22]:
bag.estimators_features_[0].shape

(5,)

# **Random Patches**


In [23]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
)

In random patches we do both rows and coloumn sapmling with replacement.

In [24]:
bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Random Patches classifier",accuracy_score(y_test,y_pred))

Random Patches classifier 0.9385


# **OOB Score**

In [27]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    oob_score=True,
    random_state=42
)

Those datapoints or samples that never come out of the bag are these and sciket learn keeps track of them can be used for testing purpose to check the accuracy.

In [28]:
bag.fit(X_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, oob_score=True, random_state=42)

In [29]:
bag.oob_score_

0.94375

In [30]:
y_pred = bag.predict(X_test)
print("Accuracy",accuracy_score(y_test,y_pred))

Accuracy 0.946


Now we are doing Grid Search CV to select the best hyperparmeters for our model.

In [31]:
from sklearn.model_selection import GridSearchCV

In [32]:
parameters = {
    'n_estimators': [50,100,500],
    'max_samples': [0.1,0.4,0.7,1.0],
    'bootstrap' : [True,False],
    'max_features' : [0.1,0.4,0.7,1.0]
    }

In [33]:
search = GridSearchCV(BaggingClassifier(), parameters, cv=5)

In [35]:
search.fit(X_train,y_train)

KeyboardInterrupt: 

In [ ]:
search.best_score_

In [ ]:
search.best_params_

# **Now we look Bagging for Regression**

In [22]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

In [34]:
housing = fetch_openml(name="california_housing", version=1)

In [35]:
X_original, y_original = housing.data, housing.target

In [36]:
X_original.shape


(20640, 9)

In [37]:
X_original.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
0,-122.23,37.88,41,880,129.0,322,126,8.3252,NEAR BAY
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,NEAR BAY
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,NEAR BAY
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,NEAR BAY
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,NEAR BAY


In [39]:
# The dataset is very large so i select only first 1000 rows.
X = X_original.iloc[:1000, :8].astype(float) # chose first 8 clouomn because 9th coloumn contains other values
y = y_original.iloc[:1000].astype(float)

In [40]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

In [41]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, y , train_size=0.80, test_size=0.20, random_state=123)
print('Train/Test Sets Sizes : ',X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)

Train/Test Sets Sizes :  (800, 8) (200, 8) (800,) (200,)


In [42]:
lr = LinearRegression()
dt = DecisionTreeRegressor()
knn = KNeighborsRegressor()

In [44]:
from sklearn.impute import SimpleImputer

# Impute missing values using the column median
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [45]:
lr.fit(X_train,Y_train)
dt.fit(X_train,Y_train)
knn.fit(X_train,Y_train)

KNeighborsRegressor()

In [46]:
y_pred1 = lr.predict(X_test)
y_pred2 = dt.predict(X_test)
y_pred3 = knn.predict(X_test)

In [47]:
print("R^2 score for LR",r2_score(Y_test,y_pred1))
print("R^2 score for DT",r2_score(Y_test,y_pred2))
print("R^2 score for KNN",r2_score(Y_test,y_pred3))

R^2 score for LR 0.6935116349999301
R^2 score for DT 0.6248327845581932
R^2 score for KNN 0.48902747772722754


**Now we will perform Bagging.**

In bagging the default parameters the estamitor is Decision Trees algo .

In [48]:
from sklearn.ensemble import BaggingRegressor

bag_regressor = BaggingRegressor(random_state=1)
bag_regressor.fit(X_train, Y_train)

BaggingRegressor(random_state=1)

In [49]:
Y_preds = bag_regressor.predict(X_test)

print('Training Coefficient of R^2 : %.3f'%bag_regressor.score(X_train, Y_train))
print('Test Coefficient of R^2 : %.3f'%bag_regressor.score(X_test, Y_test))

Training Coefficient of R^2 : 0.944
Test Coefficient of R^2 : 0.769


**Here in above cell output you can see that the accuracy has increased.**


Now we will do hyperparameter tunning using Grid Search CV to know best hyperparameter.

In [51]:
%%time

params = {'estimator': [None, LinearRegression(), KNeighborsRegressor()],
          'n_estimators': [20,50,100],
          'max_samples': [0.5,1.0],
          'max_features': [0.5,1.0],
          'bootstrap': [True, False],
          'bootstrap_features': [True, False]}

bagging_regressor_grid = GridSearchCV(BaggingRegressor(random_state=1, n_jobs=-1), param_grid =params, cv=3, n_jobs=-1, verbose=1)
bagging_regressor_grid.fit(X_train, Y_train)

print('Train R^2 Score : %.3f'%bagging_regressor_grid.best_estimator_.score(X_train, Y_train))
print('Test R^2 Score : %.3f'%bagging_regressor_grid.best_estimator_.score(X_test, Y_test))
print('Best R^2 Score Through Grid Search : %.3f'%bagging_regressor_grid.best_score_)
print('Best Parameters : ',bagging_regressor_grid.best_params_)

Fitting 3 folds for each of 144 candidates, totalling 432 fits
Train R^2 Score : 0.962
Test R^2 Score : 0.779
Best R^2 Score Through Grid Search : 0.734
Best Parameters :  {'bootstrap': True, 'bootstrap_features': False, 'estimator': None, 'max_features': 1.0, 'max_samples': 1.0, 'n_estimators': 100}
CPU times: user 1.57 s, sys: 208 ms, total: 1.78 s
Wall time: 2min 22s
